# GRPO(RLVR)推理訓練 — Qwen2.5-3B-Instruct × GSM8K

用 **GRPO**(Group Relative Policy Optimization)搭配**可驗證獎勵**(RLVR:答案對錯 +
格式,純程式判分,不用 reward model)訓練 `Qwen/Qwen2.5-3B-Instruct` 在 GSM8K 上的
數學推理,復刻 DeepSeek-R1 帶起的「completion 長度隨訓練成長、reward 同步爬升」現象。

**GRPO 一句話**:同一題抽 8 個回答,組內比較相對好壞算 advantage(取代 PPO 的 value
model),獎勵直接由規則驗證(取代 reward model)—— 便宜、不會被 reward hacking。

## 執行步驟
1. Runtime → Change runtime type → **L4 GPU**。
2. 左側 🔑 **Secrets** 加入 `HF_TOKEN`(Hugging Face write token),**並開啟此 notebook 的存取權**。
   ⚠️ 背景執行時 UI 不在場、授權彈窗無法回應 —— 務必先在 UI 掛好。
3. 保持 `SMOKE_TEST = True`,Runtime → **Run all** 跑通全流程(約 15–20 分鐘)。
4. 改 `SMOKE_TEST = False` → Run all → 關閉分頁(Colab Pro+ 背景執行,約 4–7 小時)。
   結束時自動 push 到 Hugging Face 並 `runtime.unassign()` 釋放機器。
5. 若中途斷線:重新連線後設 `RESUME = True` 再 Run all,會從 Drive 最新 checkpoint 續跑。

> 📉 **前 ~100 步 reward ≈ 0 是正常現象**(Unsloth 官方文件:等 150–300 步才開始爬),
> 不要提早砍掉健康的 run。

In [ ]:
# ====================== 參數(唯一來源,只改這裡)======================
SMOKE_TEST = True        # 先 True 跑通(~15-20 min);正式訓練改 False
RESUME = False           # True:從 Drive 最新「有效」checkpoint 續跑(中斷後使用)
PUSH_TO_HUB = True       # 訓練完 push LoRA + merged 16bit;SMOKE_TEST 時自動略過

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"   # 亦可換 unsloth/Qwen2.5-3B-Instruct 鏡像(下載較快)
LOAD_IN_4BIT = True                        # QLoRA;16bit LoRA 改 False(VRAM 需求上升)
LORA_R = 32
LORA_ALPHA = LORA_R                        # Unsloth 官方慣例:alpha = rank
LEARNING_RATE = 5e-6                       # Unsloth 官方 GRPO 範例值
MAX_STEPS = 500 if not SMOKE_TEST else 20
NUM_GENERATIONS = 8 if not SMOKE_TEST else 4
MAX_PROMPT_LENGTH = 256                    # 官方值;GSM8K 題目 + system prompt 放得下(smoke 會實測)
MAX_COMPLETION_LENGTH = 768 if not SMOKE_TEST else 256   # 官方 200;加大讓推理長度有成長空間
MAX_SEQ_LENGTH = MAX_PROMPT_LENGTH + MAX_COMPLETION_LENGTH
GPU_MEMORY_UTILIZATION = 0.85              # 官方 0.9;768-token 訓練 buffer 留 headroom
SEED = 3407                                # 官方 random_state

SAVE_STEPS = 100 if not SMOKE_TEST else 10     # checkpoint 每 100 步存一份到 Drive
SAMPLE_EVERY = 50 if not SMOKE_TEST else 5     # 每 50 步 append 2 筆完整生成到 samples.jsonl

RUN_NAME = "smoke" if SMOKE_TEST else "full"   # smoke / full 輸出完全隔離,互不污染 RESUME
DRIVE_ROOT = "/content/drive/MyDrive/grpo-rlvr-reasoning"
OUTPUT_DIR = f"{DRIVE_ROOT}/outputs_{RUN_NAME}"
SAMPLES_PATH = f"{DRIVE_ROOT}/samples_{RUN_NAME}.jsonl"
METRICS_PATH = f"{DRIVE_ROOT}/metrics_{RUN_NAME}.jsonl"

REPO_URL = "https://github.com/kuotunyu/grpo-rlvr-reasoning"  # rewards.py 的單一來源

print(f"RUN={RUN_NAME}  steps={MAX_STEPS}  gens={NUM_GENERATIONS}  completion<={MAX_COMPLETION_LENGTH}")

In [ ]:
%%capture
# 安裝方式逐字依 Unsloth 官方 nb/Qwen2.5_(3B)-GRPO.ipynb(2026-05 版)的 Colab install cell。
# ⚠️ 勿自行更動版本 pin(vllm/transformers/trl 是官方測過的組合)。
# 若安裝或 import 失敗,回官方 notebook 重抄現行 install cell:
# https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen2.5_(3B)-GRPO.ipynb
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"  # 記憶體共用的 RL standby 模式,約多 30% context
if "COLAB_" not in "".join(os.environ.keys()):
    # 非 Colab 環境:直接 pip install
    !pip install unsloth vllm
else:
    !pip install --upgrade -qqq uv
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
    !uv pip install -qqq --no-deps --upgrade "torchao>=0.16.0"
    !uv pip install transformers==4.56.2
    !uv pip install --no-deps trl==0.22.2

In [ ]:
# 確認關鍵套件版本(不 import 套件本體,避免搶在 unsloth 之前載入 transformers)
from importlib.metadata import version
for pkg in ("unsloth", "vllm", "trl", "transformers", "torch"):
    try:
        print(f"{pkg:>12} == {version(pkg)}")
    except Exception as e:
        print(f"{pkg:>12} -- {e}")

In [ ]:
# Drive 掛載 + HF_TOKEN(必須在 Run all 早期、UI 還在場時執行:
# userdata.get 在 UI 不在場的背景 session 會 TimeoutException)
import os
from google.colab import drive, userdata

drive.mount("/content/drive")
os.makedirs(f"{DRIVE_ROOT}/results/figs", exist_ok=True)

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception as e:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if not HF_TOKEN:
        raise RuntimeError(
            "讀不到 HF_TOKEN:請在 Colab Secrets 新增 HF_TOKEN(write 權限)"
            "並開啟此 notebook 的存取權;背景執行前務必先在 UI 掛好授權。"
        ) from e
os.environ["HF_TOKEN"] = HF_TOKEN
print("HF_TOKEN OK;Drive 已掛載:", DRIVE_ROOT)

In [ ]:
# 取得 rewards.py —— 與本機 pytest、eval/run_eval.py 完全同一份(單一事實來源)
import sys

if not os.path.exists("/content/grpo-rlvr-reasoning"):
    !git clone --depth 1 {REPO_URL} /content/grpo-rlvr-reasoning
sys.path.insert(0, "/content/grpo-rlvr-reasoning")

from rewards import (
    SYSTEM_PROMPT,
    correctness_reward,
    strict_format_reward,
    soft_format_reward,
    number_only_reward,
    extract_answer_block,
    extract_final_number,
    extract_gold_answer,
)
print(SYSTEM_PROMPT)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,   # QLoRA;16bit LoRA 在 PARAMS cell 改 False
    fast_inference=True,         # vLLM 加速 rollout
    max_lora_rank=LORA_R,
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION,  # OOM 時先降到 0.7
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_ALPHA,
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

In [ ]:
# GSM8K —— 只載入 train split(7,473 題)。
# 評測用的另一個 split 在整個訓練管線中零接觸(防污染;此聲明也寫進 model card)。
from datasets import load_dataset

TRAIN_SPLIT = "train"
raw = load_dataset("openai/gsm8k", "main", split=TRAIN_SPLIT)
assert len(raw) == 7473, f"train split 大小異常:{len(raw)}"

dataset = raw.map(lambda x: {
    "prompt": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": x["question"]},
    ],
    "answer": x["answer"],  # 保留原始 gold(含 '####'),rewards.extract_gold_answer 會處理
})
print(dataset)

if SMOKE_TEST:
    # 實測 tokenized prompt 長度:TRL 對過長 prompt 會「左截斷」,吃掉 system prompt
    # → 格式獎勵歸零。若 max 超過 MAX_PROMPT_LENGTH-6,把 MAX_PROMPT_LENGTH 調成 384。
    lengths = [
        len(tokenizer.apply_chat_template(p, tokenize=True, add_generation_prompt=True))
        for p in dataset.select(range(500))["prompt"]
    ]
    print(f"tokenized prompt 長度(前 500 題):max={max(lengths)}  "
          f"p99={sorted(lengths)[int(len(lengths) * 0.99)]}  上限={MAX_PROMPT_LENGTH}")
    assert max(lengths) <= MAX_PROMPT_LENGTH - 6, "prompt 超長:請把 MAX_PROMPT_LENGTH 調成 384"

In [ ]:
# 訓練中記錄:samples.jsonl(每 SAMPLE_EVERY 步 2 筆完整生成)+ metrics.jsonl(每步)
# rewards.py 保持純函數;所有 I/O 都在 notebook 端。
import functools
import json
import time

from transformers import TrainerCallback


def with_sample_logging(fn, path, every, n=2):
    """包住一個 reward function,順手把完整生成樣本 append 到 Drive。

    functools.wraps 必須保留:TRL 用 fn.__name__ 產生 metric key
    (rewards/correctness_reward/mean),名字變了曲線就斷了。
    trainer_state 是 TRL 0.24 的介面;0.22 若不傳則 fallback 到呼叫計數
    (ga=1 時每 optimizer step 呼叫一次,計數 == step)。
    """
    state = {"last_logged": -1, "calls": 0}

    @functools.wraps(fn)
    def wrapped(prompts=None, completions=None, answer=None, trainer_state=None, **kwargs):
        scores = fn(prompts=prompts, completions=completions, answer=answer, **kwargs)
        state["calls"] += 1
        step = trainer_state.global_step if trainer_state is not None else state["calls"]
        if step % every == 0 and step != state["last_logged"]:
            state["last_logged"] = step
            try:
                with open(path, "a", encoding="utf-8") as f:
                    for i in range(min(n, len(completions))):
                        c = completions[i]
                        text = c[-1]["content"] if isinstance(c, list) else str(c)
                        p = prompts[i]
                        question = p[-1]["content"] if isinstance(p, list) else str(p)
                        f.write(json.dumps({
                            "step": step,
                            "time": time.time(),
                            "question": question,
                            "completion": text,
                            "extracted": extract_final_number(extract_answer_block(text) or ""),
                            "gold": extract_gold_answer(answer[i]) if answer else None,
                            "correctness": scores[i],
                        }, ensure_ascii=False) + "\n")
            except Exception as e:  # 記錄失敗不影響訓練
                print("sample logging failed (non-fatal):", e)
        return scores

    return wrapped


logged_correctness_reward = with_sample_logging(
    correctness_reward, SAMPLES_PATH, SAMPLE_EVERY, n=2
)


class JsonlMetricsCallback(TrainerCallback):
    """每次 on_log 把 logs append 到 Drive 的 metrics.jsonl。

    curves 資料即時落地 —— runtime 中途被收走,曲線也還在。
    """

    def __init__(self, path):
        self.path = path

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        try:
            with open(self.path, "a", encoding="utf-8") as f:
                f.write(json.dumps({"step": state.global_step, **logs}, ensure_ascii=False) + "\n")
        except Exception as e:
            print("metrics logging failed (non-fatal):", e)

In [ ]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    use_vllm=True,
    learning_rate=LEARNING_RATE,        # 官方 5e-6
    adam_beta1=0.9,
    adam_beta2=0.99,                    # 官方
    weight_decay=0.001,                 # 官方
    warmup_ratio=0.1,                   # 官方
    lr_scheduler_type="cosine",         # 官方
    optim="adamw_8bit",                 # 官方
    logging_steps=1,                    # 每步記錄,曲線才有解析度
    per_device_train_batch_size=1,      # 官方
    gradient_accumulation_steps=1,      # 官方(GRPO 一步 = 一輪完整 rollout,ga=4 等於 4 倍時間)
    num_generations=NUM_GENERATIONS,
    max_prompt_length=MAX_PROMPT_LENGTH,
    max_completion_length=MAX_COMPLETION_LENGTH,
    max_steps=MAX_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,                 # Drive 上最多留 3 份 checkpoint(LoRA-only,每份 < 0.5GB)
    max_grad_norm=0.1,                  # 官方
    seed=SEED,
    report_to="none",
    output_dir=OUTPUT_DIR,              # 直接寫 Drive:背景 session 隨時可能被收走
)
# 刻意不設(沿用 TRL 預設,與官方 notebook 相同):beta=0.0(不載 ref model,省 VRAM)、
# temperature、loss_type、scale_rewards。
# ⚠️ remove_unused_columns 必須維持預設 False,否則 `answer` 欄位進不了 reward functions,
#    correctness 會永遠是 0。

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        soft_format_reward,
        strict_format_reward,
        number_only_reward,
        logged_correctness_reward,
    ],
    args=training_args,
    train_dataset=dataset,
    callbacks=[JsonlMetricsCallback(METRICS_PATH)],
)

In [ ]:
# 續跑閘門 + 開訓
import glob
import re as _re


def latest_valid_checkpoint(output_dir):
    """從最新往回找第一個「完整」的 checkpoint。

    比 transformers 的 get_last_checkpoint 多一層防呆:Drive FUSE 上被
    中途收走的 session 可能留下半寫的 checkpoint 目錄。
    """
    if not os.path.isdir(output_dir):
        return None
    ckpts = [
        p for p in glob.glob(os.path.join(output_dir, "checkpoint-*"))
        if _re.fullmatch(r"checkpoint-\d+", os.path.basename(p))
    ]
    for c in sorted(ckpts, key=lambda p: int(p.rsplit("-", 1)[-1]), reverse=True):
        has_state = os.path.exists(os.path.join(c, "trainer_state.json"))
        has_weights = any(
            os.path.exists(os.path.join(c, w))
            for w in ("adapter_model.safetensors", "model.safetensors")
        )
        if has_state and has_weights:
            return c
    return None


ckpt = latest_valid_checkpoint(OUTPUT_DIR) if RESUME else None
if RESUME and ckpt is None:
    print("RESUME=True 但 Drive 上沒有可用 checkpoint —— 改為從頭訓練")
print("resume_from_checkpoint =", ckpt)

trainer_stats = trainer.train(resume_from_checkpoint=ckpt)
print(trainer_stats)

import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)  # VRAM 實測紀錄

In [ ]:
# 兩張招牌圖:reward 曲線、completion 長度曲線
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt


def load_metrics(path):
    rows = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            try:
                r = json.loads(line)
            except json.JSONDecodeError:
                continue  # 被收走瞬間的半行
            if "step" in r:
                rows[r["step"]] = r  # RESUME 會產生重複 step:保留最後一筆
    return [rows[k] for k in sorted(rows)]


def resolve_key(rows, candidates, pattern):
    """metric key 在 trl 版本間會漂移 —— 依候選清單 + regex fallback 解析。"""
    keys = set()
    for r in rows[: min(50, len(rows))]:
        keys.update(r.keys())
    for c in candidates:
        if c in keys:
            return c
    for k in sorted(keys):
        if _re.match(pattern, k):
            return k
    raise KeyError(f"找不到 metric key;候選={candidates};現有 keys={sorted(keys)}")


rows = load_metrics(METRICS_PATH)
reward_key = resolve_key(rows, ["reward"], r"^rewards?$")
length_key = resolve_key(
    rows,
    ["completions/mean_length", "completion_length", "completions/mean_terminated_length"],
    r"^completions?/.*length",
)
print("resolved keys:", reward_key, "|", length_key)


def rolling_mean(ys, w):
    out = []
    for i in range(len(ys)):
        lo = max(0, i - w + 1)
        out.append(sum(ys[lo : i + 1]) / (i - lo + 1))
    return out


FIG_DIRS = [f"{DRIVE_ROOT}/results/figs", "/content/grpo-rlvr-reasoning/results/figs"]
for d in FIG_DIRS:
    os.makedirs(d, exist_ok=True)

for key, title, fname, ylabel in [
    (reward_key, "GRPO on GSM8K - mean reward per step", "reward_curve.png",
     "mean reward (max 3.5)"),
    (length_key, "GRPO on GSM8K - mean completion length", "completion_length_curve.png",
     "completion length (tokens)"),
]:
    xs = [r["step"] for r in rows if key in r]
    ys = [r[key] for r in rows if key in r]
    plt.figure(figsize=(8, 4.5), dpi=150)
    plt.plot(xs, ys, alpha=0.3, linewidth=0.8, label="per step")
    w = max(1, min(20, len(ys) // 5))
    plt.plot(xs, rolling_mean(ys, w), linewidth=1.8, label=f"rolling mean (w={w})")
    plt.xlabel("step")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    for d in FIG_DIRS:
        plt.savefig(os.path.join(d, fname))
    plt.close()
    print("saved:", fname, "->", FIG_DIRS)

In [ ]:
# 訓練前 vs 訓練後對照樣本(model card 素材)—— 固定 2 題 train 題
from vllm import SamplingParams

LORA_DIR = f"{DRIVE_ROOT}/grpo_saved_lora_{RUN_NAME}"
model.save_lora(LORA_DIR)

DEMO_QUESTIONS = [dataset[0]["question"], dataset[1]["question"]]
demo_sampling = SamplingParams(temperature=0.8, top_p=0.95, max_tokens=1024)  # 官方示範參數


def generate_demo(lora_request=None):
    outs = []
    for q in DEMO_QUESTIONS:
        text = tokenizer.apply_chat_template(
            [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": q},
            ],
            tokenize=False,
            add_generation_prompt=True,
        )
        out = model.fast_generate(text, sampling_params=demo_sampling, lora_request=lora_request)
        outs.append(out[0].outputs[0].text)
    return outs


base_samples = generate_demo(lora_request=None)                          # 訓練前(base + LoRA 未載入)
trained_samples = generate_demo(lora_request=model.load_lora(LORA_DIR))  # 訓練後(GRPO LoRA)

for q, b, t in zip(DEMO_QUESTIONS, base_samples, trained_samples):
    print("=" * 80)
    print("Q:", q)
    print("--- base ---")
    print(b)
    print("--- GRPO ---")
    print(t)

In [ ]:
# Push 到 Hugging Face + model cards + 伺服器端驗證
if SMOKE_TEST and PUSH_TO_HUB:
    PUSH_TO_HUB = False
    print("SMOKE_TEST 模式:跳過 push")

PUSH_VERIFIED = False
if PUSH_TO_HUB:
    from huggingface_hub import HfApi, ModelCard, whoami

    HF_USERNAME = whoami(token=HF_TOKEN)["name"]
    LORA_REPO = f"{HF_USERNAME}/qwen2.5-3b-grpo-gsm8k-lora"
    MERGED_REPO = f"{HF_USERNAME}/qwen2.5-3b-grpo-gsm8k"

    # 1) 先推便宜的 LoRA adapter —— merge 途中掛掉也至少留得住成果
    model.push_to_hub_merged(LORA_REPO, tokenizer, save_method="lora", token=HF_TOKEN)
    # 2) merged 16bit(~6GB 先在 /content 本地磁碟落地再上傳;不要寫 Drive)
    model.push_to_hub_merged(MERGED_REPO, tokenizer, save_method="merged_16bit", token=HF_TOKEN)

    api = HfApi(token=HF_TOKEN)
    for repo in (LORA_REPO, MERGED_REPO):
        for fig in ("reward_curve.png", "completion_length_curve.png"):
            api.upload_file(
                path_or_fileobj=f"{DRIVE_ROOT}/results/figs/{fig}",
                path_in_repo=f"figs/{fig}",
                repo_id=repo,
            )

    def clip(text, n=1200):
        return text if len(text) <= n else text[:n] + "\n...(截斷)"

    def build_model_card(repo_id, is_lora):
        kind = "LoRA adapter(rank 32)" if is_lora else "merged 16-bit 完整模型"
        library = "peft" if is_lora else "transformers"
        pair_note = (
            f"完整 merged 模型:[{MERGED_REPO}](https://huggingface.co/{MERGED_REPO})"
            if is_lora
            else f"LoRA adapter 版本:[{LORA_REPO}](https://huggingface.co/{LORA_REPO})"
        )
        samples_md = ""
        for q, b, t in zip(DEMO_QUESTIONS, base_samples, trained_samples):
            samples_md += (
                f"\n**題目**:{q}\n\n"
                f"<details><summary>訓練前(base)</summary>\n\n```\n{clip(b)}\n```\n</details>\n"
                f"<details><summary>訓練後(GRPO)</summary>\n\n```\n{clip(t)}\n```\n</details>\n"
            )
        header = (
            "---\n"
            "license: apache-2.0\n"
            "base_model: Qwen/Qwen2.5-3B-Instruct\n"
            "datasets:\n"
            "- openai/gsm8k\n"
            "language:\n"
            "- en\n"
            f"library_name: {library}\n"
            "pipeline_tag: text-generation\n"
            "tags:\n"
            "- grpo\n"
            "- rlvr\n"
            "- reasoning\n"
            "- unsloth\n"
            "- trl\n"
            "- qwen2.5\n"
            "---\n"
        )
        body = f"""
# Qwen2.5-3B GRPO(RLVR)on GSM8K — {kind}

以 **GRPO**(Group Relative Policy Optimization)+ **可驗證獎勵**(RLVR)在 GSM8K
數學題上訓練 `Qwen/Qwen2.5-3B-Instruct` 的推理能力。{pair_note}。
訓練程式與獎勵函數:[GitHub — grpo-rlvr-reasoning]({REPO_URL})。

## 方法(白話)

GRPO 對同一題一次抽 {NUM_GENERATIONS} 個回答,**組內互相比較**算出每個回答的
相對優勢(advantage),取代 PPO 的 value model;獎勵不是另一個神經網路
(reward model),而是**可程式驗證的規則**:

| 獎勵函數 | 條件 | 分數 |
|---|---|---|
| correctness_reward | `<answer>` 內數字 == 標準答案 | 2.0 |
| strict_format_reward | 完整 `<reasoning>...</reasoning><answer>...</answer>` 結構 | 0.5 |
| soft_format_reward | 兩組 tag 依序出現(部分符合) | 0.5 |
| number_only_reward | `<answer>` 是純數字 | 0.5 |

答案對錯是可驗證的 —— 不會被 reward hacking、也省掉訓 reward model 的成本。
這就是 DeepSeek-R1 帶起的 RLVR 路線;本專案復刻其招牌現象:
**completion 長度隨訓練成長、reward 同步爬升**(模型自己學會寫更長的推理)。

## 訓練曲線

![reward curve](figs/reward_curve.png)

![completion length curve](figs/completion_length_curve.png)

## 訓練前後對照
{samples_md}

## 超參數

| 項目 | 值 |
|---|---|
| base model | {MODEL_NAME} |
| 演算法 | GRPO(TRL + Unsloth,vLLM rollout) |
| LoRA rank / alpha | {LORA_R} / {LORA_ALPHA}(QKVO + MLP 全模組) |
| learning rate | {LEARNING_RATE}(cosine,warmup 0.1,adamw_8bit) |
| num_generations | {NUM_GENERATIONS} |
| max prompt / completion length | {MAX_PROMPT_LENGTH} / {MAX_COMPLETION_LENGTH} |
| steps | {MAX_STEPS} |
| 量化 | {'4-bit QLoRA' if LOAD_IN_4BIT else '16-bit LoRA'}(訓練時) |
| seed | {SEED} |

超參以 Unsloth 官方 GRPO 範例為基準;偏差:LoRA r=32(官方 64)、
completion 上限 768(官方 200,為觀察長度成長而加大)、
strict_format regex 修正了官方版缺 re.DOTALL 導致多行推理永不匹配的問題。

## 資料與污染聲明

只使用 `openai/gsm8k`(config `main`)的 **train split(7,473 題)**;
評測用的另一個 split 在整個訓練管線中**零接觸**(notebook 內有 assert 與
程式級保證),評測結果見 GitHub repo 的 `results/eval_report.md`。

## License

Apache-2.0
"""
        return header + body

    for repo, is_lora in ((LORA_REPO, True), (MERGED_REPO, False)):
        ModelCard(build_model_card(repo, is_lora)).push_to_hub(repo, token=HF_TOKEN)

    def verify_push():
        """伺服器端驗證(不只「沒丟例外」):兩個 repo 的關鍵檔案都要在。"""
        lora_files = set(api.list_repo_files(LORA_REPO))
        merged_files = set(api.list_repo_files(MERGED_REPO))
        ok = {"adapter_config.json", "adapter_model.safetensors", "README.md"} <= lora_files
        ok &= "config.json" in merged_files and "README.md" in merged_files
        ok &= any(f.endswith(".safetensors") for f in merged_files)
        ok &= "figs/reward_curve.png" in merged_files
        return ok

    PUSH_VERIFIED = verify_push()
    print("PUSH_VERIFIED =", PUSH_VERIFIED)
    print(f"https://huggingface.co/{LORA_REPO}")
    print(f"https://huggingface.co/{MERGED_REPO}")

In [ ]:
# 最終閘門:只有「push 已在伺服器端驗證成功」才釋放機器
if PUSH_VERIFIED and not SMOKE_TEST:
    print("push 已驗證 —— flush Drive 後釋放 runtime")
    drive.flush_and_unmount()  # 確保 checkpoint / samples / metrics 都已落盤
    from google.colab import runtime
    runtime.unassign()
else:
    reason = "SMOKE_TEST 模式" if SMOKE_TEST else "push 未驗證成功(檢查上個 cell 的輸出)"
    print(f"不釋放 runtime:{reason}。絕不自動關掉還沒把成果推上去的機器。")

## 下一步

訓練完成、模型已 push 之後:

1. 到 Hugging Face 確認兩個 repo(LoRA 與 merged)的 model card 與曲線圖。
2. 跑評測(Colab 或本機 GPU 皆可):
   ```bash
   python eval/run_eval.py --trained-model <HF_USERNAME>/qwen2.5-3b-grpo-gsm8k
   ```
3. 把 `results/`(兩張圖 + eval_report.md + 逐題 jsonl)commit 回 GitHub repo,
   README 的曲線圖與對照表就會亮起來。